# Epi Info AI Chi Square for Trend validation lab — V0.15

Compare the candidate Rust/WebAssembly Extended Mantel-Haenszel linear-trend result with an independent Python formula. Passing is evidence, not statistical approval.

In [ ]:
import math
from pyodide.http import pyfetch
from js import WebAssembly, Uint8Array
fixture_response = await pyfetch('/validation-fixtures/chi-square-trend-v0.15.json')
fixture_response.raise_for_status(); fixture = await fixture_response.json()
wasm_response = await pyfetch('/epi2x2.wasm')
instance = await WebAssembly.instantiate(Uint8Array.new(await wasm_response.buffer()), {})
rust = instance.instance.exports

In [ ]:
rows = fixture['input']['rows']
rust.trend_reset()
for index, row in enumerate(rows):
    assert rust.trend_set_row(index, row['score'], row['cases'], row['controls']) == 1
rust_result = {
    'oddsRatios': [float(rust.trend_odds_ratio(i, len(rows))) for i in range(len(rows))],
    'chiSquare': float(rust.trend_chi_square(len(rows))),
    'pValue': float(rust.trend_p_value(len(rows))),
}
rust_result

In [ ]:
scores = [row['score'] for row in rows]
cases = [row['cases'] for row in rows]
controls = [row['controls'] for row in rows]
totals = [a + b for a, b in zip(cases, controls)]
n1, n2 = sum(cases), sum(controls); n = n1 + n2
t1 = sum(a*x for a, x in zip(cases, scores))
t2 = sum(m*x for m, x in zip(totals, scores))
t3 = sum(m*x*x for m, x in zip(totals, scores))
variance = n1*n2*(n*t3 - t2*t2)/(n*n*(n-1))
python_chi = (t1 - n1*t2/n)**2/variance
python_p = math.erfc(math.sqrt(python_chi/2))
python_odds = [(a*controls[0])/(b*cases[0]) for a, b in zip(cases, controls)]
tolerance = fixture['tolerances']['absolute']
assert abs(rust_result['chiSquare'] - python_chi) <= tolerance
assert abs(rust_result['pValue'] - python_p) <= tolerance
assert all(abs(a-b) <= tolerance for a, b in zip(rust_result['oddsRatios'], python_odds))
assert rust_result['pValue'] < fixture['expected']['pValueUpperBound']
{'status': 'PASS', 'python': {'chiSquare': python_chi, 'pValue': python_p, 'oddsRatios': python_odds}, 'rust': rust_result}